# NeuroScan — Stage 2: Glioma Subtype Classifier

Trains the deployed `glioma_subtype_classifier.pth` (4-class EfficientNet-B0) that runs in the cascade after stage 1 predicts `glioma`.

**Source:** `fernando2rad/brain-tumor-mri-images-44c` only. For each subtype, all three MRI sequences (T1, T2, T1C+) are merged into a single class folder, then split 80/20 train/test at the image level with `random.seed(42)`.

**Output:** `glioma_subtype_classifier.pth` → place in `backend/`.

**Class index order (alphabetical, from `ImageFolder`):**
`[astrocytoma, ependymoma, glioblastoma, oligodendroglioma]`
This MUST match `GLIOMA_SUBTYPES` in `backend/model.py`.

Run on a Colab GPU runtime. Set the `KAGGLE_API_TOKEN` Colab Secret before running.

**Known methodological limitations** (see `ARCHITECTURE.md`):
- Patient-level data leakage from random image-level 80/20 split
- T1/T2/T1C+ sequences of the same patient go in both train and test
- Test set doubles as validation set
- Same data44 images may also be in stage 1's train set as `glioma`

## 1. Setup & Download data44

In [ ]:
!pip install -q kaggle

In [ ]:
import os
from google.colab import userdata

# Kaggle token from Colab Secrets (left sidebar key icon).
# Add a secret named KAGGLE_API_TOKEN before running this cell.
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download -d fernando2rad/brain-tumor-mri-images-44c
!unzip -q brain-tumor-mri-images-44c.zip -d data44
print("Done")

## 2. Build glioma subtype dataset

Merge T1/T2/T1C+ sequences per subtype, 80/20 split with seed 42.

In [ ]:
import os, shutil, random

GLIOMA_SUBTYPES = {
    'astrocytoma':       ['Astrocitoma T1',       'Astrocitoma T2',       'Astrocitoma T1C+'],
    'glioblastoma':      ['Glioblastoma T1',      'Glioblastoma T2',      'Glioblastoma T1C+'],
    'oligodendroglioma': ['Oligodendroglioma T1', 'Oligodendroglioma T2', 'Oligodendroglioma T1C+'],
    'ependymoma':        ['Ependimoma T1',        'Ependimoma T2',        'Ependimoma T1C+'],
}

# Clean slate
shutil.rmtree('glioma_subtypes', ignore_errors=True)
for split in ['train', 'test']:
    for cls in GLIOMA_SUBTYPES:
        os.makedirs(f'glioma_subtypes/{split}/{cls}', exist_ok=True)

# Find the actual data directory (44c dataset is sometimes nested)
data_root = 'data44'
for item in os.listdir(data_root):
    sub = os.path.join(data_root, item)
    if os.path.isdir(sub) and not item.startswith('.'):
        if any(os.path.isdir(os.path.join(sub, d)) for d in os.listdir(sub)):
            data_root = sub
            break

idx = 0
for cls, folders in GLIOMA_SUBTYPES.items():
    images = []
    for folder in folders:
        folder_path = os.path.join(data_root, folder)
        if os.path.exists(folder_path):
            for f in os.listdir(folder_path):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    images.append(os.path.join(folder_path, f))

    random.seed(42)
    random.shuffle(images)
    split_idx = int(len(images) * 0.8)

    for i, img_path in enumerate(images):
        split = 'train' if i < split_idx else 'test'
        dst = f'glioma_subtypes/{split}/{cls}/{idx}_{os.path.basename(img_path)}'
        shutil.copy2(img_path, dst)
        idx += 1

print("Glioma subtypes dataset:")
for split in ['train', 'test']:
    print(f"\n  {split}:")
    for cls in sorted(os.listdir(f'glioma_subtypes/{split}')):
        n = len(os.listdir(f'glioma_subtypes/{split}/{cls}'))
        print(f"    {cls}: {n} images")

## 3. Train glioma subtype classifier — EfficientNet-B0, fully unfrozen, batch 16, 20 epochs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('glioma_subtypes/train', transform=train_transforms)
test_dataset = datasets.ImageFolder('glioma_subtypes/test', transform=test_transforms)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"\nClasses: {train_dataset.classes}")
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(1280, 4),
)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model.train()
    running_loss = correct = total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    test_acc = 100.0 * correct / total
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f"Glioma {epoch+1}/20 | Loss: {train_loss:.4f} | Train: {train_acc:.1f}% | Test: {test_acc:.1f}% | LR: {lr:.6f}")

torch.save(model.cpu().state_dict(), 'glioma_subtype_classifier.pth')
from google.colab import files
files.download('glioma_subtype_classifier.pth')